# 1242. Web Crawler Multithreaded

Implement `crawl(startUrl, htmlParser)` and return all URLs reachable from `startUrl` that belong to the same hostname.

Notes for this draft notebook:
- Output order is not required.
- Traversal should stay within the same host as `startUrl`.
- `htmlParser.getUrls(url)` returns the outgoing links for a page.

What you do each time you reach a URL:
1. Take one URL from your work queue.
2. Call `htmlParser.getUrls(current_url)` to get all outgoing links from that page.
3. For each returned URL:
   - Extract its hostname.
   - If the hostname is different from `startUrl`'s hostname, ignore it.
   - If you have already visited it, ignore it.
   - Otherwise mark it as visited and add it to the queue so it will be processed later.
4. Continue until there are no more same-host unvisited URLs left.

Sample expectation on the first example:
- Start at `http://news.yahoo.com/news/topics/`.
- Visit it and call `getUrls(...)`.
- You receive:
  - `http://news.yahoo.com`
  - `http://news.yahoo.com/news`
  - `http://news.google.com`
- Check each one:
  - `http://news.yahoo.com` -> same hostname (`news.yahoo.com`), not seen before, so add it.
  - `http://news.yahoo.com/news` -> same hostname, not seen before, so add it.
  - `http://news.google.com` -> different hostname, so skip it.
- Next, when you reach `http://news.yahoo.com`, call `getUrls(...)` again.
- If it points back to URLs you already visited or already queued, do not add duplicates.
- Next, when you reach `http://news.yahoo.com/news`, do the same check again.
- Final result for that sample should contain only:
  - `http://news.yahoo.com/news/topics/`
  - `http://news.yahoo.com`
  - `http://news.yahoo.com/news`

Threading note:
- Multiple workers may process different queued URLs at the same time.
- The shared `visited` set / queue update must be protected so the same URL is not added twice.


In [1]:
def test():
    class MockHtmlParser:
        def __init__(self, graph):
            self.graph = graph

        def getUrls(self, url):
            return self.graph.get(url, [])

    cases = [
        ((
            'http://news.yahoo.com/news/topics/',
            MockHtmlParser({
                'http://news.yahoo.com/news/topics/': [
                    'http://news.yahoo.com',
                    'http://news.yahoo.com/news',
                    'http://news.google.com'
                ],
                'http://news.yahoo.com': [
                    'http://news.yahoo.com/news/topics/',
                    'http://news.yahoo.com/news'
                ],
                'http://news.yahoo.com/news': [
                    'http://news.yahoo.com/news/topics/',
                    'http://news.yahoo.com'
                ],
                'http://news.google.com': [
                    'http://news.google.com/world'
                ],
            }),
        ), [
            'http://news.yahoo.com/news/topics/',
            'http://news.yahoo.com',
            'http://news.yahoo.com/news',
        ]),
        ((
            'http://example.org/a',
            MockHtmlParser({
                'http://example.org/a': ['http://example.org/b', 'http://other.org/x'],
                'http://example.org/b': ['http://example.org/c'],
                'http://example.org/c': [],
                'http://other.org/x': ['http://other.org/y'],
            }),
        ), [
            'http://example.org/a',
            'http://example.org/b',
            'http://example.org/c',
        ]),
        ((
            'http://site.com/home',
            MockHtmlParser({
                'http://site.com/home': [],
            }),
        ), [
            'http://site.com/home',
        ]),
    ]
    for i, (args, expected) in enumerate(cases, 1):
        got = sorted(Solution().crawl(*args))
        expected = sorted(expected)
        assert got == expected, f'case {i}: expected {expected}, got {got}'


In [17]:
from threading import Thread, lock
from concurrent.futures import ThreadPoolExecutor
from queue import Queue 
import concurrent.futures
import urllib.parse


class Solution:
    def __init__(self, max_workers=10, max_queue_size=10):
        self.max_workers = max_workers
        self.max_queue_size = max_queue_size
        self.reset()
    
    def reset(self):
        self.seen = set()
        self.queue = Queue(maxsize=self.max_queue_size)
        self.executor = ThreadPoolExecutor(max_workers=self.max_workers)
        self.lock = lock()
    
    def worker(self, htmlParser):
        
        while True:
            # take one URL from work queue
            item = self.queue.get()

            if item is None:
                break
            
            #item is url we can check it's children from htmlparser 
            next_urls = htmlParser[item]
            for url in next_urls:
                self.add_url_to_queue(urls)
            self.queue.task_done()

    def add_url_to_queue(self, url):
        # atomically check queue 
        with self.lock:
            if url in self.seen:
                return False
            else:
                try:
                    self.queue.put_nowait(url)
                except:
                    # queue is full
                    return False

                self.seen.add(url)
                return True
            
    def crawl(self, startUrl, htmlParser):

        with self.executor as executor:
            for _ in range(4):
                executor.submit(self.worker, htmlParser)
            self.queue.put(startUrl)
            self.queue.join()

            for _ in range(4):
                self.queue.put(None)

            self.reset()
        

        


ImportError: cannot import name 'lock' from 'threading' (/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/threading.py)

## Critique of Current Attempt

### 1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.

Your current direction is aiming at the standard queue-centric worker model, which is a strong fit for the problem. The intended complexity target is driven by how many URLs you actually process and how many edges you inspect, but your current draft is not yet in a state where its real complexity can be trusted because the worker loop, duplicate control, shutdown behavior, and parser usage are still incomplete.

Main trade-off in your design so far:
- Good: explicit shared queue, explicit worker pool, explicit shared `seen` state.
- Risky: too much state is living on `self` without a clear lifecycle reset boundary.
- Risky: bounded queue introduces backpressure, but for this problem it also creates avoidable failure paths if producers temporarily outpace consumers.
- Risky: correctness currently depends on housekeeping details more than on the traversal idea itself.

### 2. Critique of the problem-solving approach, including progression of thought and method.

The core instinct is good practice: choose one work abstraction (`Queue`), one deduplication abstraction (`seen`), and one concurrency abstraction (`ThreadPoolExecutor`). That is the right shape. Where the design weakens is that responsibilities are not separated enough yet.

What looks good:
- You recognized this is fundamentally a work-dispatch problem.
- You are trying to protect shared mutation with a lock.
- You are thinking about worker reuse instead of spawning unbounded threads.

What needs tightening:
- `htmlParser` should be treated as an interface, not as a dictionary-like store.
- The exact atomic boundary is still unclear: is "check seen + record seen + enqueue" one protected unit or several?
- `seen` persistence should be scoped to one `crawl(...)` call unless you can prove instance reuse is safe.
- Executor ownership and queue ownership should have one clear startup and one clear shutdown path.
- Naming drift (`excutor` vs `executor`, `lock` import usage) suggests the design has not been pressure-tested yet.

On your specific question about persisting post-processed results: for this LeetCode problem, the useful persisted result is usually just the visited URL set for the current crawl. You generally do not want a second long-lived storage layer unless you are modeling a production crawler. If you do separate states mentally, the usual buckets are:
- discovered but not yet processed
- processed already
- final returned same-host URLs

In a clean design for this problem, at least two of those can often be derived from each other instead of being stored independently. Extra persistence is usually a source of bugs here, not clarity.

### 3. Improvements to Algorithm (Hint-Only Guidance, no full solution code)

Use these checkpoints to improve the design without jumping to code:

1. Define the state machine for one URL precisely:
   `new -> scheduled -> being processed -> fully processed`.
   Which transitions need synchronization, and which do not?

2. Decide whether a URL should enter `seen` when it is enqueued or when a worker starts processing it.
   Ask yourself: which choice better prevents duplicate scheduling?

3. Challenge the bounded queue decision.
   If the queue is full, is "drop this URL" ever logically correct for this problem?

4. Write down the exact shutdown condition before coding it.
   Is termination based on queue emptiness, all scheduled work completing, explicit sentinels, or a combination?

5. Separate orchestration from page-processing in your head.
   One unit should answer: "how do workers live and stop?"
   Another should answer: "what do I do with one URL once I have it?"

6. Ask whether any result container besides `seen` is truly necessary.
   If you keep both `seen` and `processed`, what invariant makes both worth the complexity?

### 4. Applications in real-life situations, including AI-agent and engineering potential applications in 2026. Include examples from big tech and startups (frontier tech) for the exact problem and the generalized pattern. Be critical and outline tradeoffs, when to use this algorithm/design, and when not to use it.

Transferable systems pattern: this is a concurrent frontier-expansion workflow over a graph of discovered work items.

Literal usage vs analogy:
- Literal: internal site crawling, dependency graph expansion, reachable-resource discovery.
- Partial analogy: agent tool-call fan-out, document graph traversal, service dependency inspection.

What is its usefulness in designing large-scale data-driven applications?
- It is useful when new work is discovered incrementally during processing.
- It gives a clear place to apply deduplication, backpressure, and parallelism.
- It becomes fragile when completion detection and state ownership are vague.

Concrete examples:
- Big-tech-scale infrastructure: a search indexing pipeline may discover linked internal resources and schedule recrawl tasks while deduplicating by canonical URL. This is a direct pattern match.
- Startup / frontier-tech: a retrieval startup may traverse citation links, project docs, or knowledge-graph edges to expand context candidates for downstream ranking. This is a partial pattern match.

AI-agent mapping in 2026:
- Plausible use: an agent platform fans out tool calls to inspect reachable documents, APIs, or repos, while deduplicating already-enqueued work items. This is a conceptual transfer, not the exact same problem.
- Do not use this approach if the agent workflow is mostly linear and latency-sensitive with only a handful of known steps; queue-centric concurrency would add coordination overhead for little gain.

Concise application case:
- Context and constraint: a repo analysis service must discover reachable documentation pages under one domain with strict duplicate avoidance.
- Pattern choice: bounded worker pool plus shared deduplicated frontier.
- Decision and expected outcome: stable concurrency, controlled memory growth, and deterministic no-duplicate scheduling if the shared-state boundary is designed correctly.

```mermaid
flowchart LR
    Seed[Initial work item] --> Frontier[(Shared frontier)]
    Frontier --> WorkerA[Worker A]
    Frontier --> WorkerB[Worker B]
    WorkerA --> Discover[Discover new reachable items]
    WorkerB --> Discover
    Discover --> Dedup[(Dedup gate)]
    Dedup --> Frontier
    Dedup --> Sink[Accepted results]
```

When to use this design:
- dynamic work discovery
- medium to high IO wait
- strong need to prevent duplicate scheduling

When not to use it:
- tiny workloads where single-thread traversal is simpler
- workloads where all tasks are known upfront
- AI-agent flows where ordering and auditability matter more than throughput

### 5. Open Questions to Challenge My Understanding (non-spoiler). Ask 3-6 targeted questions tied to likely blind spots from my solution and reasoning.

1. In your current design, what exact event makes a URL eligible to enter `seen`: discovery, successful enqueue, or start of processing, and why?
2. If the queue is full at the moment a valid same-host URL is discovered, what should happen logically for correctness?
3. What invariant should hold between `seen`, queue contents, and the final returned results at every point in time?
4. If a worker receives a stop signal too early, what unfinished-work scenario becomes possible in your current structure?
5. Why should crawl-specific mutable state probably be initialized in `crawl(...)` rather than only in `__init__`?
6. In your draft, are you optimizing for throughput, correctness, or API elegance first, and does the code structure reflect that priority clearly?

### 6. Next-Step Application Challenges (Similar but Variant) with Learning-Goal Intent. Provide 2-4 concise challenge prompts that are close to the current problem but differ in one key dimension (constraints, interface, mutability, streaming, memory, distributed setting, etc.). For each challenge include:

1. Variant: same-host crawl, but each page fetch may fail transiently.
   Learning goal intent: reason about retries without breaking deduplication invariants.
   What changed from the original problem: page processing is no longer guaranteed to succeed once.
   Why this change matters for design decisions: you must decide whether failed work stays "seen", gets retried, or moves to a separate state.

2. Variant: all reachable URLs are known upfront in one batch before processing starts.
   Learning goal intent: compare frontier-expansion design against fixed-task parallel processing.
   What changed from the original problem: no dynamic discovery during processing.
   Why this change matters for design decisions: a shared work queue may still work, but the crawler-style architecture is less necessary.

3. Variant: memory is capped tightly, so you cannot keep all seen URLs in RAM forever.
   Learning goal intent: think about approximate deduplication and externalized state.
   What changed from the original problem: state size becomes a first-class constraint.
   Why this change matters for design decisions: persistence and false-positive/false-negative tradeoffs become central.

4. Variant: crawl is distributed across multiple machines.
   Learning goal intent: identify which local invariants break once queue and deduplication are no longer process-local.
   What changed from the original problem: shared state becomes remote and coordination cost dominates.
   Why this change matters for design decisions: lock-based correctness no longer scales; ownership and idempotency become the main concerns.


## Core Patterns Reference

These diagrams focus on the concurrency patterns and decision points that matter for this problem, without showing implementation.

### 1. Single URL processing loop

```mermaid
flowchart TD
    A[Take next URL from queue] --> B[Fetch outgoing links]
    B --> C{Another link to inspect}
    C -- Yes --> D[Read hostname]
    D --> E{Same host}
    E -- No --> C
    E -- Yes --> G{Already seen}
    G -- Yes --> C
    G -- No --> H[Mark seen]
    H --> I[Enqueue URL]
    I --> C
    C -- No --> J[Done with this URL]
```

### 2. Shared-state coordination pattern

```mermaid
flowchart LR
    W1[Worker 1] --> L[Critical section]
    W2[Worker 2] --> L
    W3[Worker 3] --> L
    L --> S[(visited / seen set)]
    L --> Q[(shared work queue)]
    S --> D1[Prevent duplicate scheduling]
    Q --> D2[Distribute remaining work]
```

Use this pattern for the check-and-add step:
- check whether a URL was already seen
- if not, record it exactly once
- schedule it exactly once

### 3. Worker lifecycle pattern

```mermaid
stateDiagram-v2
    [*] --> Waiting
    Waiting --> Processing: receive URL
    Processing --> Discovering: call getUrls
    Discovering --> Scheduling: same-host unseen links found
    Scheduling --> Waiting: more work may exist
    Discovering --> Waiting: nothing new found
    Waiting --> Stopping: receive stop signal / shutdown condition
    Stopping --> [*]
```

### 4. End-to-end work expansion

```mermaid
flowchart TD
    S[startUrl] --> Q1[Queue]
    Q1 --> W[Workers]
    W --> U1[Same-host URL A]
    W --> U2[Same-host URL B]
    W --> X[Other-host URL]
    U1 --> Q1
    U2 --> Q1
    X --> Drop[Discard]
```

### 5. Mental checklist per URL

- Who owns taking the next URL from the queue?
- Where is hostname filtering applied?
- Where is duplicate prevention enforced?
- What exact step must be atomic?
- How do workers know there is no more work?
- How do workers stop cleanly after all queued work is done?


## Common Architectural Variants

### 17. Variant A - Queue-Centric

Workers continuously pull from a shared queue. This is the most common design for this problem.

```mermaid
flowchart LR
    Seed[startUrl] --> Q[(Shared queue)]
    Q --> W1[Worker 1]
    Q --> W2[Worker 2]
    Q --> W3[Worker 3]
    W1 --> P[Fetch and inspect links]
    W2 --> P
    W3 --> P
    P --> S[(Shared seen set)]
    S --> Q
```

Very common because:
- the work unit is explicit
- backpressure is easier to control
- shutdown logic is easier to reason about

### 17. Variant B - Recursive Futures

Tasks recursively spawn more tasks. This can feel elegant, but termination is harder.

```mermaid
flowchart TD
    T0[Task for startUrl] --> T1[Spawn task for URL A]
    T0 --> T2[Spawn task for URL B]
    T1 --> T3[Spawn task for URL C]
    T1 --> T4[Spawn task for URL D]
    T2 --> T5[Spawn task for URL E]
    T3 --> S[(Shared seen set)]
    T4 --> S
    T5 --> S
```

Hard parts:
- deduplicate before spawning
- track when the full task tree is complete
- avoid runaway task creation

### 17. Variant C - Async Event Loop

No threads. Instead use cooperative coroutines, an async queue, and one event loop.

```mermaid
flowchart LR
    Loop[Event loop] --> AQ[(Async queue)]
    AQ --> C1[Coroutine 1]
    AQ --> C2[Coroutine 2]
    AQ --> C3[Coroutine 3]
    C1 --> IO[Await IO work]
    C2 --> IO
    C3 --> IO
    IO --> S[(Shared seen set)]
    S --> AQ
```

Useful when:
- the environment already uses async patterns
- blocking calls can be made awaitable
- you want concurrency without thread coordination


In [25]:
from threading import Thread, Lock
from concurrent.futures import ThreadPoolExecutor
from queue import Queue 
import concurrent.futures
from urllib.parse import urlparse

class Solution:
    def __init__(self, max_queue_size=10):
        self.max_queue_size = max_queue_size
        self.seen = set()
        self.queue = Queue()
        self.executor = ThreadPoolExecutor()
        self.lock = Lock()
        self.start_host = ''
    
    def reset(self, startUrl):
        self.seen = set()
        self.queue = Queue()
        self.executor = ThreadPoolExecutor()
        self.lock = Lock()
        self.start_host = urlparse(startUrl).netloc

    
    def check_url(self,url):
        return urlparse(url).netloc == self.start_host

    def worker(self, htmlParser):
        
        while True:
            # take one URL from work queue
            item = self.queue.get()
            if item is None:
                break
            
            #item is url we can check it's children from htmlparser 
            next_urls = htmlParser[item]
            for url in next_urls:
                self.add_url_to_queue(url)
            
            self.queue.task_done()

    def add_url_to_queue(self, url):
        # atomically check queue 
        with self.lock:
            if url not in self.seen and self.check_url(url):
                self.seen.add(url)
                self.queue.put(url)
                return True
        return False
            
    def crawl(self, startUrl, htmlParser):
        self.reset(startUrl)

        with self.executor as executor:
            for _ in range(4):
                executor.submit(self.worker, htmlParser)
            self.queue.put(startUrl)

            for _ in range(4):
                self.queue.put(None)
            self.queue.join()

        return self.seen
        

In [ ]:
# When Solution().crawl is runnable, run the harness below.
# If this cell appears to hang, the current worker/queue shutdown logic is likely blocking:
# workers call queue.get() forever, sentinels are queued before all real work is drained,
# and each worker uses htmlParser like a dict instead of calling htmlParser.getUrls(url).
test()
print('PASS')


## Producer-Consumer Blocking and Canonical Solutions

### 8. This is called producer-consumer cyclic blocking

This failure mode happens when the same worker threads are responsible for both:
- consuming from the queue, and
- producing newly discovered work back into that same bounded queue.

If all workers are busy processing items and each tries to enqueue more work into a full queue, they can all block before any thread returns to `get()`.

This is often called:
- producer-consumer cyclic blocking
- feedback deadlock
- bounded-buffer deadlock

The key issue is the cycle:
- workers need queue space to continue,
- but queue space only appears if some worker resumes consuming,
- and no worker can resume consuming because they are all blocked producing.

### 9. Why unbounded queues avoid this

With an unbounded queue, `enqueue` does not block due to capacity.

So a worker can:
- finish processing the current URL,
- append discovered URLs,
- return to consuming the next item.

That makes liveness much easier to reason about. The system can still be slow or memory-heavy, but it is much less likely to deadlock due to queue capacity alone.

### 10. So why use bounded queues at all?

Because unbounded queues trade liveness simplicity for resource risk.

| Problem | Meaning |
| --- | --- |
| memory explosion | frontier can grow without limit |
| latency blowup | queue backlog gets huge and old work waits too long |
| loss of backpressure | producers can outrun the rest of the system |

So bounded queues are:
- safer for resources,
- harder for liveness.

This is a classic systems tradeoff: resource control versus simpler progress guarantees.

### 11. Canonical solutions

Real systems solve this several ways.

**Solution A - Queue much larger than worker count**

Very common practical trick.

```mermaid
flowchart LR
    Q[(Large bounded queue)] --> W1[Worker 1]
    Q --> W2[Worker 2]
    Q --> W3[Worker 3]
    W1 --> P1[Process URL and discover more]
    W2 --> P2[Process URL and discover more]
    W3 --> P3[Process URL and discover more]
    P1 --> Q
    P2 --> Q
    P3 --> Q
    N[Capacity much larger than worker count] --> Q
```

Invariant:
- workers are unlikely to saturate the queue simultaneously.

This is not a formal guarantee, but in many production workloads it is enough.

**Solution B - Separate producer/consumer roles**

Instead of having the same worker both consume and produce, you separate:
- frontier expansion,
- queue insertion,
- queue consumption.

```mermaid
flowchart LR
    FE[Frontier expansion threads] --> B[Discovery buffer]
    B --> PI[Queue insertion stage]
    PI --> Q[(Bounded work queue)]
    Q --> C1[Consumer worker 1]
    Q --> C2[Consumer worker 2]
    C1 --> FE
    C2 --> FE
```

That changes the topology so some threads are always available to drain work.

**Solution C - Non-blocking enqueue with retry/defer**

Workers avoid permanent blocking on `put()`.

```mermaid
flowchart TD
    W[Worker processing URL] --> D[Discover next URLs]
    D --> T{queue has space}
    T -- yes --> Q[(Bounded queue)]
    T -- no --> B[Temporary local buffer]
    B --> R[Retry later or defer]
    R --> T
    Q --> NX[Next consumer]
```

Instead they may:
- try `put_nowait()`,
- buffer discovered URLs temporarily,
- retry later or hand work off elsewhere.

This improves liveness, but it makes correctness and retry policy more complex.

**Solution D - Work stealing / local queues**

```mermaid
flowchart LR
    L1[(Worker 1 local queue)] --> W1[Worker 1]
    L2[(Worker 2 local queue)] --> W2[Worker 2]
    L3[(Worker 3 local queue)] --> W3[Worker 3]
    W1 --> L1
    W2 --> L2
    W3 --> L3
    W3 -. steal .-> L1
    W2 -. steal .-> L3
```

Workers enqueue locally first instead of contending on one global bounded queue for every URL.

That reduces central queue pressure and often improves throughput, especially when discoveries are bursty.

**Solution E - Async/event-driven architecture**

Instead of blocking OS threads, workers yield control when waiting.

```mermaid
flowchart LR
    EL[Event loop] --> AQ[(Async queue)]
    AQ --> C1[Coroutine 1]
    AQ --> C2[Coroutine 2]
    C1 --> IO1[Await fetch or parse]
    C2 --> IO2[Await fetch or parse]
    IO1 --> AQ
    IO2 --> AQ
    IO1 -. yield .-> EL
    IO2 -. yield .-> EL
```

This is often more scalable for high-concurrency crawlers, but it is also more complex to design and reason about.

### 12. Important realization

Bounded queues are not free safety.

They introduce capacity constraints into the execution topology.

Once that happens, scheduling and liveness become part of correctness, not just performance.


# Decision matrix for safety

| Pattern            | Complexity  | Liveness Safety | Throughput   | Memory Safety | Best For                   | Main Weakness             |
| ------------------ | ----------- | --------------- | ------------ | ------------- | -------------------------- | ------------------------- |
| A Large Queue      | Very Low    | Probabilistic   | Good         | Moderate      | interviews/simple crawlers | no formal guarantee       |
| B Separate Roles   | Medium      | Strong          | Good         | Strong        | pipelines/services         | more architecture         |
| C Retry/Defer      | Medium-High | Moderate        | Moderate     | Strong        | bursty systems             | retry complexity          |
| D Work Stealing    | High        | Strong          | Excellent    | Strong        | high-performance runtimes  | implementation complexity |
| E Async/Event Loop | High        | Strong          | Excellent IO | Strong        | huge IO fanout             | cognitive complexity      |


## Why `urlparse(...).netloc` is used

```python
start_host = urlparse(startUrl).netloc
if urlparse(url).netloc != start_host:
    continue
```

This keeps the crawler on the same hostname as `startUrl`.

Example:

- `startUrl = "http://news.yahoo.com/news/topics/"`
- `urlparse(startUrl).netloc` gives `"news.yahoo.com"`

Now suppose we discover these URLs:

1. `http://news.yahoo.com/news`
   - `urlparse(...).netloc` is `"news.yahoo.com"`
   - same host, so we keep it

2. `http://news.google.com/world`
   - `urlparse(...).netloc` is `"news.google.com"`
   - different host, so we skip it

So this check means:

- crawl URLs from the same hostname
- ignore URLs from other websites


1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.

Your overall direction is converging toward the right systems shape: shared frontier queue, shared visited set, small fixed worker pool. That is the standard interview pattern for this problem. The last attempt is better than the earlier one because hostname filtering is now explicit via `urlparse(...).netloc`, and the state reset is at least scoped to `crawl(startUrl, htmlParser)` instead of being half-global.

The main issue is that the last attempt is still not meaningfully analyzable by its intended complexity because it does not currently complete correctly. In the ideal version of this pattern, the cost is roughly proportional to the number of same-host URLs visited plus the number of edges inspected. In your current version, liveness dominates complexity analysis: if the pipeline can stall or terminate workers too early, asymptotic discussion stops mattering because the algorithm is not reliably producing a full traversal.

Trade-offs across the attempts:
- First attempt: stronger instinct for bounded backpressure, but much more fragile. Bounded queues inside a feedback loop are easy to deadlock or drop work unless the producer-consumer contract is precise.
- Last attempt: simpler queue semantics and simpler host filtering, which is an improvement.
- Last attempt downside: shutdown and work-accounting are still incorrect enough that the crawler can appear to "take forever" even on tiny tests, because the coordination protocol is not yet sound.

2. Critique of the problem-solving approach, including progression of thought and method.

Your progression is good in one important way: you did not jump to recursion or unbounded thread spawning. You correctly modeled this as a concurrent work-dispatch problem. That is the right instinct for both interviews and real systems.

The main weakness is that your reasoning has focused more on "how do I set up threads and queue objects?" than on "what exact lifecycle state transitions must always be true?" In concurrent crawlers, the hard part is rarely the traversal idea. It is proving these invariants:
- when a URL becomes `seen`
- when a URL is guaranteed to be processed
- when workers are allowed to stop
- when the main thread is allowed to return

Your notebook shows healthy iteration:
- You moved from a very rough draft toward a more interface-shaped solution.
- You introduced host filtering, which matches the problem contract.
- You scoped reset logic closer to a single crawl call.

But the current final cell still reveals a key reasoning gap: you are using concurrency primitives before fully specifying the work protocol. That is why the notebook feels "slow" now. It is not slow in the normal performance sense. It is waiting because the system does not yet have a correct completion condition.

3. Improvements to Algorithm (Hint-Only Guidance, no full solution code)

1. Ask yourself: when should `startUrl` enter `seen`?
If it is only placed into the queue but never recorded as visited up front, what result do you expect when the crawl finishes?

2. Look at the line inside `worker` that retrieves neighbors.
What is the contract of `htmlParser` in the problem statement: dictionary access or method call? If the interface is wrong, what exact runtime behavior do you expect?

3. Focus on shutdown ordering.
Should workers receive `None` before or after you know that all discovered real URLs have been fully processed? What happens if a worker consumes a sentinel while another real URL is still waiting or will be enqueued shortly after?

4. Examine `queue.join()` and `task_done()` as a pair.
For every `queue.get()`, including sentinel paths, is the unfinished-task counter guaranteed to be decremented correctly? If not, what symptom would you see in the notebook?

5. Treat this as a state machine exercise.
Write the states for one URL as:
`unseen -> enqueued -> dequeued -> expanded -> done`
Then ask: under your current locking and queue logic, can any URL skip a state, repeat a state, or get counted in `seen` without ever being expanded?

6. Before thinking about speed, prove liveness on the smallest case.
For the one-node test case `http://site.com/home`, can you narrate exactly which thread does each step and why the system is guaranteed to terminate?

4. Applications in real-life situations, including AI-agent and engineering potential applications in 2026. Include examples from big tech and startups (frontier tech) for the exact problem and the generalized pattern. Be critical and outline tradeoffs, when to use this algorithm/design, and when not to use it.

The transferable systems pattern is concurrent frontier expansion with deduplication and bounded shared state. Literal usage is partial: production systems do crawl graphs, route work through queues, and deduplicate discovered items. The exact LeetCode solution is not itself a production crawler design, but the queue-plus-visited-set pattern is conceptually and partially transferable.

What is its usefulness in designing large-scale data-driven applications? It is useful when a system must discover new work from prior work, avoid duplicate processing, and maintain throughput under I/O latency. That shows up in web indexing, dependency discovery, graph exploration, knowledge synchronization, retrieval precomputation, and agent task expansion. The boundary is important: large systems add persistence, retries, leases, distributed ownership, observability, and fault tolerance. The interview version stops far before that.

Big-tech-scale example:
A search infrastructure service may assign URL partitions across many workers, with a distributed frontier and a dedup layer. The literal pattern maps directly at a high level, but the real system needs cross-machine coordination, storage-backed queues, politeness budgets, and failure recovery.

Startup/frontier-tech example:
A document-ingestion platform for enterprise copilots may recursively discover linked docs in SaaS systems, queue fetch tasks, deduplicate connectors' outputs, and keep traversal scoped to a tenant boundary. That is a close conceptual match: not a public web crawler, but the same "discover, filter, dedup, expand" pattern.

A plausible 2026 AI-agent application:
An agent orchestration system might expand a plan graph by discovering new tool calls or subgoals from prior tool outputs, while deduplicating equivalent tasks and constraining expansion to the same project or tenant boundary. That is a conceptual transfer, not a literal one. Do not use this exact approach when agent tasks require durable distributed coordination, retries, cancellation propagation, and fairness across many users; a single in-memory queue on one process is the wrong reliability model.

Concise application case:
Context and constraint: a frontier-tech team is building a repo-mapping agent that must follow imports and internal links across 50,000 files without revisiting nodes or crossing repository boundaries.
Algorithm/pattern choice: concurrent frontier traversal with shared deduplication and boundary filtering.
Decision and expected outcome: use a worker pool plus a thread-safe visited set to parallelize metadata fetches while preventing duplicate expansion; expected outcome is much higher throughput than serial traversal, but only if shutdown/accounting semantics are proven correct.

```mermaid
flowchart LR
    S[Seed item] --> Q[(Shared frontier queue)]
    Q --> W1[Worker 1]
    Q --> W2[Worker 2]
    Q --> W3[Worker 3]
    W1 --> D[Discover neighbors]
    W2 --> D
    W3 --> D
    D --> F[Boundary filter]
    F --> V{Already visited?}
    V -- No --> Q
    V -- Yes --> X[Drop duplicate]
```

When to use this design:
- The work graph expands dynamically from discovered neighbors.
- Each unit of work is mostly I/O-bound.
- Duplicate suppression is essential.
- Boundary filtering is simple and cheap.

When not to use this design:
- You need durable recovery across crashes.
- The frontier can exceed process memory.
- Work ownership is distributed across many machines.
- Cancellation, retries, and fairness matter more than raw local throughput.

Counterexample in AI-agent systems:
Do not use this exact in-memory worker-queue design for long-running multi-tenant autonomous agents coordinating thousands of tool calls across hours or days. You need persistent orchestration, leases, idempotency keys, and resumability.

5. Open Questions to Challenge My Understanding (non-spoiler). Ask 3-6 targeted questions tied to likely blind spots from my solution and reasoning.

1. Under your current code, what exact event tells the main thread that no future same-host URL can still be discovered?
2. If a worker receives `None` before another worker finishes expanding a real URL, what guarantees that newly discovered URLs will still be processed?
3. Why should "check host", "check visited", and "enqueue" be reasoned about as one atomic workflow rather than three unrelated steps?
4. In your current implementation, does `seen` mean "already processed" or "already scheduled"? Which interpretation makes termination easier to reason about, and why?
5. Your notebook feels like it is "taking so long." Can you distinguish whether that symptom comes from poor throughput, deadlock, premature worker exit, or unfinished-task accounting?
6. If `htmlParser` is an interface object, what assumptions become invalid when you access it like `htmlParser[item]`?

6. Next-Step Application Challenges (Similar but Variant) with Learning-Goal Intent. Provide 2-4 concise challenge prompts that are close to the current problem but differ in one key dimension (constraints, interface, mutability, streaming, memory, distributed setting, etc.). For each challenge include:
   - Learning goal intent
   - What changed from the original problem
   - Why this change matters for design decisions

1. Variant: Same crawler, but `getUrls(url)` can fail transiently and should be retried up to 2 times.
Learning goal intent: reason about retries without breaking deduplication or termination.
What changed from the original problem: unreliable interface instead of always-successful blocking calls.
Why this change matters for design decisions: you must decide whether `seen` means attempted, scheduled, or successfully expanded.

2. Variant: Same-host crawl, but the queue is memory-bounded and the graph may contain 10 million URLs.
Learning goal intent: understand backpressure and liveness tradeoffs.
What changed from the original problem: memory pressure is now a first-class constraint.
Why this change matters for design decisions: bounded queues and durable spillover change both correctness and throughput strategy.

3. Variant: Distributed crawl across 20 workers on different machines.
Learning goal intent: separate local concurrency from distributed ownership.
What changed from the original problem: shared memory is gone.
Why this change matters for design decisions: deduplication, leases, and completion detection can no longer rely on one in-process set and queue.

4. Variant: Agent task expansion instead of URLs, where each task can spawn more tasks but tasks can also be canceled.
Learning goal intent: transfer the frontier-expansion pattern into 2026 agent orchestration.
What changed from the original problem: tasks have lifecycle control and cancellation semantics.
Why this change matters for design decisions: completion logic must account for tasks that disappear, retry, or are superseded, not just tasks that finish normally.


In [ ]:
from threading import Thread, Lock
from concurrent.futures import ThreadPoolExecutor
from queue import Queue 
import concurrent.futures
from urllib.parse import urlparse

class Solution:
    def __init__(self, max_queue_size=10):
        self.max_queue_size = max_queue_size
        self.seen = set()
        self.queue = Queue()
        self.executor = ThreadPoolExecutor()
        self.lock = Lock()
        self.start_host = ''
    
    def reset(self, startUrl):
        self.seen = set()
        self.queue = Queue()
        self.executor = ThreadPoolExecutor()
        self.lock = Lock()
        self.start_host = urlparse(startUrl).netloc

    
    def check_url(self,url):
        return urlparse(url).netloc == self.start_host

    def worker(self, htmlParser):
        
        while True:
            # take one URL from work queue
            item = self.queue.get()
            if item is None:
                break
            
            #item is url we can check it's children from htmlparser 
            next_urls = htmlParser.getUrls(item)
            for url in next_urls:
                self.add_url_to_queue(url)
            
            self.queue.task_done()

    def add_url_to_queue(self, url):
        # atomically check queue 
        with self.lock:
            print(f"checking if url: {url} in seen/ valid")
            
            if url not in self.seen and self.check_url(url):
                print(f"placing url in queue: {url}")
                self.seen.add(url)
                self.queue.put(url)
                return True
        return False
            
    def crawl(self, startUrl, htmlParser):
        self.reset(startUrl)

        with self.executor as executor:
            for _ in range(4):
                executor.submit(self.worker, htmlParser)
            self.queue.put(startUrl)
            self.seen.add(startUrl)

            for _ in range(4):
                self.queue.put(None)
            self.queue.join()

        return self.seen
        

In [7]:
# When Solution().crawl is runnable, run the harness below.
# If this cell appears to hang, the current worker/queue shutdown logic is likely blocking:
# workers call queue.get() forever, sentinels are queued before all real work is drained,
# and each worker uses htmlParser like a dict instead of calling htmlParser.getUrls(url).
test()
print('PASS')


checking if url: http://news.yahoo.com in seen/ valid
placing url in queue: http://news.yahoo.com
checking if url: http://news.yahoo.com/news in seen/ valid
placing url in queue: http://news.yahoo.com/news
checking if url: http://news.google.com in seen/ valid


KeyboardInterrupt: 

You are extremely close. The reason it “takes a long time” is not performance — it is a **lifecycle / queue accounting deadlock**.

You already built the correct topology:

* persistent workers,
* shared frontier,
* atomic dedup,
* poison-pill shutdown.

Now the remaining issue is:

```text id="c0"
termination protocol correctness
```

---

# 1. Core Bug

This ordering is wrong:

```python id="b1"
for _ in range(4):
    self.queue.put(None)

self.queue.join()
```

Conceptually you are saying:

```text id="b2"
shutdown workers BEFORE all work completion is guaranteed
```

That creates a liveness hazard.

---

# 2. Why this is dangerous

Workers consume from one mixed stream:

```text id="b3"
[real work items] + [termination signals]
```

But now:

* sentinels can be consumed early,
* workers terminate prematurely,
* remaining URLs never processed.

Then:

```text id="b4"
queue.join()
```

waits forever because:

* unfinished task count never reaches zero.

Classic producer-consumer shutdown bug.

---

# 3. Queue accounting invariant you are violating

Very important:

```text id="b5"
every queue.put()
must eventually pair with
exactly one task_done()
```

But your worker:

```python id="b6"
if item is None:
    break
```

exits WITHOUT:

```python id="b7"
self.queue.task_done()
```

So:

* unfinished task counter leaks.

This alone can deadlock `join()` forever.

This is probably the main reason for your hang.

---

# 4. The hidden Queue invariant

Internally queue tracks roughly:

```text id="b8"
unfinished_tasks += put()
unfinished_tasks -= task_done()
```

`join()` blocks until:

```text id="b9"
unfinished_tasks == 0
```

Your poison pills increase unfinished count,
but never decrement it.

---

# 5. Correct lifecycle shape conceptually

The canonical lifecycle is:

```text id="b10"
start workers
→ enqueue seed
→ dynamically expand frontier
→ wait until all real work completed
→ enqueue shutdown sentinels
→ workers terminate
→ executor exits
```

Very important ordering.

---

# 6. Another subtle issue

Inside lock:

```python id="b11"
with self.lock:
    ...
    self.queue.put(url)
```

Right now queue is unbounded so mostly fine.

But conceptually:

```text id="b12"
blocking operations inside critical sections are dangerous
```

You had previously improved this correctly,
then regressed slightly.

---

# 7. Another conceptual issue

You reset:

```python id="b13"
self.executor = ThreadPoolExecutor()
```

inside `reset()`.

Then:

```python id="b14"
with self.executor as executor:
```

means executor lifecycle tightly coupled to crawl invocation.

Not necessarily wrong,
but architecturally odd because:

* executor ownership becomes mutable object state.

Cleaner design usually:

* executor local to crawl lifecycle.

---

# 8. Important positive sign

This is now correct:

```python id="b15"
next_urls = htmlParser.getUrls(item)
```

You fixed the abstraction boundary:

* parser as expansion oracle,
  not dictionary.

Good systems decomposition improvement.

---

# 9. Your graph traversal semantics are now mostly correct

You now properly implemented:

```text id="b16"
discover
→ filter
→ dedup
→ enqueue
```

This is the canonical concurrent frontier invariant.

---

# 10. Your filtering abstraction is also correct

This:

```python id="b17"
urlparse(url).netloc == self.start_host
```

is a proper:

```text id="b18"
graph traversal admissibility predicate
```

Very good.

---

# 11. Deep interpretation of your current bug

You currently have:

```text id="b19"
termination edges entering execution graph
before frontier exhaustion is globally proven
```

In concurrency terms:

* shutdown is racing against work generation.

---

# 12. The mental model you need

Think of queue state as:

| Queue item type | Meaning                  |
| --------------- | ------------------------ |
| URL             | expandable frontier node |
| None            | worker-termination token |

The crucial invariant:

```text id="b20"
termination tokens may only enter frontier
after expansion frontier reaches fixpoint
```

Very elegant way to think about it.

---

# 13. Why the bug feels confusing

Because logically:

```text id="b21"
you think "I already added all workers"
```

But workers themselves dynamically generate:

* more work,
* after shutdown tokens already inserted.

So system becomes:

```text id="b22"
mixed work/termination race
```

---

# 14. Your current architecture maturity

You are now beyond:

* primitive syntax learning.

You are now debugging:

```text id="b23"
distributed lifecycle coordination invariants
```

That is a meaningful jump in systems thinking.

---

# 15. Remaining major concepts to internalize

| Concept                     | Why it matters             |
| --------------------------- | -------------------------- |
| queue accounting invariants | prevents hangs             |
| shutdown ordering           | preserves liveness         |
| fixpoint detection          | defines completion         |
| sentinel semantics          | orderly worker termination |
| lifecycle topology          | determines correctness     |

---

# 16. The deepest compression

Your crawler is fundamentally:

```text id="final"
a dynamically expanding concurrent graph traversal
whose correctness depends on proving
global frontier exhaustion before termination propagation
```

That’s the real conceptual structure underneath this problem.


In [8]:
from threading import Thread, Lock
from concurrent.futures import ThreadPoolExecutor
from queue import Queue 
import concurrent.futures
from urllib.parse import urlparse

class Solution:
    def __init__(self, max_queue_size=10, max_workers = 4):
        self.max_queue_size = max_queue_size
        self.seen = set()
        self.queue = Queue()
        self.lock = Lock()
        self.start_host = ''
        self.max_workers = max_workers
    
    def reset(self, startUrl):
        self.seen = set()
        self.queue = Queue()
        self.lock = Lock()
        self.start_host = urlparse(startUrl).netloc

    
    def check_url(self,url):
        return urlparse(url).netloc == self.start_host

    def worker(self, htmlParser):
        
        while True:
            # take one URL from work queue
            item = self.queue.get()
            try:
                if item is None:
                    break
                
                #item is url we can check it's children from htmlparser 
                next_urls = htmlParser.getUrls(item)
                for url in next_urls:
                    self.add_url_to_queue(url)

            finally: 
                self.queue.task_done()

    def add_url_to_queue(self, url):
        # atomically check queue 
        valid = False
        with self.lock:
            print(f"checking if url: {url} in seen/ valid")
            
            if url not in self.seen and self.check_url(url):
                print(f"placing url in queue: {url}")
                self.seen.add(url)
                valid = True
                
        #separation of two blocking operations.
        if valid:
            self.queue.put(url)
            return True
        else:
            return False
            
    def crawl(self, startUrl, htmlParser):
        self.reset(startUrl)

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            for _ in range(self.max_workers):
                executor.submit(self.worker, htmlParser)
            self.queue.put(startUrl)
            self.seen.add(startUrl)
            
            self.queue.join()

            for _ in range(self.max_workers):
                self.queue.put(None)

        return self.seen
        

In [9]:
# When Solution().crawl is runnable, run the harness below.
# If this cell appears to hang, the current worker/queue shutdown logic is likely blocking:
# workers call queue.get() forever, sentinels are queued before all real work is drained,
# and each worker uses htmlParser like a dict instead of calling htmlParser.getUrls(url).
test()
print('PASS')


checking if url: http://news.yahoo.com in seen/ valid
placing url in queue: http://news.yahoo.com
checking if url: http://news.yahoo.com/news in seen/ valid
placing url in queue: http://news.yahoo.com/news
checking if url: http://news.google.com in seen/ valid
checking if url: http://news.yahoo.com/news/topics/ in seen/ valid
checking if url: http://news.yahoo.com/news in seen/ valid
checking if url: http://news.yahoo.com/news/topics/ in seen/ valid
checking if url: http://news.yahoo.com in seen/ valid
checking if url: http://example.org/b in seen/ valid
placing url in queue: http://example.org/b
checking if url: http://other.org/x in seen/ valid
checking if url: http://example.org/c in seen/ valid
placing url in queue: http://example.org/c
PASS


# Sentinel Nodes must wait for invariant:

```unfinished_tasks == 0```
which is basically global counter between queue and workers.

```mermaid
flowchart TD

    A[Start Workers]
    --> B[Insert Seed URL]

    B --> C[Workers Expand Frontier]

    C --> D[queue.join waits for unfinished_tasks == 0]

    D --> E[Insert N Sentinels]

    E --> F[Workers Consume Sentinel]

    F --> G[Workers Exit]

    G --> H[Executor Shutdown]
```

You fixed ONE bug with `try/finally`, but the OTHER bug still exists.

These are actually two separate invariants:

| Problem                   | Fix                    |
| ------------------------- | ---------------------- |
| queue accounting leak     | `finally: task_done()` |
| premature worker shutdown | shutdown ordering      |

You solved the first.
You still have the second.

That distinction is the key thing you're missing.

---

# 1. What `try/finally` fixed

Previously you had:

```python id="p1"
item = self.queue.get()

if item is None:
    break

self.queue.task_done()
```

Problem:

```text id="p2"
None exits loop before task_done()
```

So:

* queue internal unfinished counter never decremented.

Meaning:

| Queue operation       | Counter  |
| --------------------- | -------- |
| `put(None)`           | +1       |
| missing `task_done()` | never -1 |

So `join()` waits forever.

---

# 2. Why `finally` fixes this

Now you have:

```python id="p3"
item = self.queue.get()

try:
    if item is None:
        break
finally:
    self.queue.task_done()
```

Critical property:

```text id="p4"
finally always executes
```

Even if:

* break,
* return,
* exception,
* continue.

So now:

* every successful `get()`
* guarantees exactly one `task_done()`.

Excellent.

You repaired:

```text id="p5"
queue accounting symmetry
```

---

# 3. But the SECOND bug still exists

THIS is still wrong:

```python id="p6"
for _ in range(self.num_workers):
    self.queue.put(None)

self.queue.join()
```

You are still inserting:

* termination signals
  BEFORE:
* work completion proven.

That is a completely different issue.

---

# 4. Why this is dangerous

Imagine runtime order like this:

---

## Initial state

Queue:

```text id="p7"
[startUrl, None, None, None, None]
```

Workers:

* all start simultaneously.

---

# 5. Possible schedule

Worker 1:

* gets `startUrl`
* begins expansion

Worker 2:

* immediately gets `None`
* exits

Worker 3:

* gets `None`
* exits

Worker 4:

* gets `None`
* exits

Now only:

* ONE worker remains alive.

---

# 6. Worse case

Suppose worker 1 discovers:

```text id="p8"
1000 child URLs
```

Now:

* only one worker processes everything,
* concurrency collapses.

Still maybe works.

---

# 7. Catastrophic case

Suppose instead:

Queue:

```text id="p9"
[startUrl, None, None, None]
```

Worker 1:

* gets `startUrl`

Workers 2,3,4:

* consume all sentinels,
* terminate.

Worker 1:

* later discovers children,
* eventually consumes last sentinel,
* exits.

Now:

* newly enqueued descendants remain forever unprocessed.

No workers alive.

---

# 8. Visualizing the race

```mermaid id="m1"
sequenceDiagram

    participant Main
    participant W1
    participant W2
    participant Queue

    Main->>Queue: put(startUrl)
    Main->>Queue: put(None x4)

    W1->>Queue: get(startUrl)

    W2->>Queue: get(None)
    W2->>W2: terminate

    W1->>Queue: enqueue(children)

    W1->>Queue: get(None)
    W1->>W1: terminate

    Note over Queue: children still remain
```

THIS is the real bug.

Not queue accounting anymore.

---

# 9. Deep interpretation

You are violating:

```text id="p10"
termination causality ordering
```

Shutdown is entering execution graph:

* before descendant generation stabilizes.

---

# 10. The key conceptual difference

## Queue accounting problem

Question:

```text id="p11"
Did every get() pair with task_done()?
```

Solved by:

* `finally`.

---

## Shutdown ordering problem

Question:

```text id="p12"
Did termination happen AFTER all reachable descendants were generated?
```

NOT solved by:

* `finally`.

Completely separate issue.

---

# 11. Correct lifecycle topology

The correct ordering conceptually is:

```mermaid id="m2"
flowchart TD

    A[Start Workers]
    --> B[Insert Seed]

    B --> C[Expand Graph Dynamically]

    C --> D[Wait queue.join]

    D --> E[Inject Sentinels]

    E --> F[Workers Exit]
```

Meaning:

```text id="p13"
join BEFORE sentinels
```

not after.

---

# 12. Why `join()` must come first

Because:

```python id="p14"
queue.join()
```

proves:

```text id="p15"
unfinished_tasks == 0
```

Meaning:

* no queued items,
* no active expansions,
* no future descendants pending.

THEN:

* safe to terminate workers.

---

# 13. Another subtle issue in your code

This line is wrong:

```python id="p16"
with ThreadPoolExecutor(num_workers=self.num_workers)
```

Correct parameter name is:

```python id="p17"
max_workers=
```

Pure syntax issue.

---

# 14. Structural critique of your current design

Now your architecture is actually pretty solid.

---

# 15. Strong parts

| Component             | Assessment        |
| --------------------- | ----------------- |
| persistent workers    | good              |
| queue-driven frontier | canonical         |
| dedup invariant       | correct           |
| lock scope            | good              |
| task accounting       | now correct       |
| parser abstraction    | good              |
| same-host filter      | good              |
| blocking queue        | correct primitive |

This is genuinely strong concurrency structure now.

---

# 16. Remaining architectural issue

The ONLY major correctness issue left is:

```text id="p18"
termination propagation enters system too early
```

That’s it.

You are now debugging:

* lifecycle topology,
  not:
* primitive misuse.

That is a substantial improvement.

---

# 17. Deepest mental model

Your system now has two distinct flows:

---

## Work propagation flow

```text id="p19"
URL
→ expansion
→ new URLs
→ frontier growth
```

---

## Lifecycle propagation flow

```text id="p20"
sentinel
→ worker termination
→ reduced processing capacity
→ global shutdown
```

Correctness requires:

```text id="p21"
lifecycle flow must causally trail work flow
```

That is the deepest insight here.


# Concurrent Life Cycle Topologies:

These 5 concepts are deeply connected. Together they form the core correctness framework for concurrent worker systems.

The unifying abstraction is:

```text id="u0"
maintaining globally consistent progress
in a partially ordered evolving execution graph
```

---

# 1. Queue Accounting Invariants

## Core idea

```text id="q0"
The system must maintain an exact conservation law for work units.
```

Queue accounting answers:

```text id="q1"
How do we know whether work still exists?
```

---

# 2. Formal invariant

Python Queue internally maintains:

```text id="q2"
unfinished_tasks
```

Invariant:

```text id="q3"
unfinished_tasks
=
(# puts)
-
(# task_done calls)
```

Completion iff:

```text id="q4"
unfinished_tasks == 0
```

---

# 3. Why this matters

Without strict accounting:

| Failure               | Result               |
| --------------------- | -------------------- |
| missing `task_done()` | permanent hang       |
| extra `task_done()`   | counter corruption   |
| lost work item        | premature completion |
| duplicate accounting  | inconsistent state   |

---

# 4. Queue accounting visualization

```mermaid id="m1"
flowchart LR

    P[Producer put]
    -->|+1 unfinished| Q[Queue]

    Q --> C[Consumer get]

    C --> D[Process Task]

    D --> T[task_done]

    T -->|-1 unfinished| Z[Completion Detector]
```

---

# 5. Deep interpretation

This is really:

```text id="q5"
distributed resource accounting
```

Very similar to:

* garbage collection,
* distributed reference counting,
* transaction tracking.

---

# 6. Shutdown Ordering

## Core idea

```text id="s0"
Termination must not overtake unfinished causality.
```

Shutdown ordering answers:

```text id="s1"
When is it safe to terminate workers?
```

---

# 7. Dangerous ordering

Bad:

```mermaid id="m2"
flowchart TD

    A[Work Still Expanding]
    --> B[Sentinel Inserted Early]

    B --> C[Workers Exit]

    C --> D[Unprocessed Descendants]

    D --> E[Deadlock / Lost Work]
```

---

# 8. Correct ordering

Correct:

```mermaid id="m3"
flowchart TD

    A[Expand Frontier]
    --> B[All Work Completed]

    B --> C[Inject Sentinels]

    C --> D[Workers Exit]

    D --> E[Executor Shutdown]
```

---

# 9. Why this matters

Shutdown is itself:

```text id="s2"
a causality event
```

So shutdown must obey:

* dependency ordering,
* frontier exhaustion.

Otherwise:

* termination races with work generation.

---

# 10. Fixpoint Detection

This is one of the deepest concepts.

## Core idea

```text id="f0"
Completion means no future state transitions are possible.
```

---

# 11. Why queue empty is insufficient

Suppose:

| Queue | Worker                   |
| ----- | ------------------------ |
| empty | currently expanding node |

Worker may still:

* discover descendants,
* enqueue more work.

So:

```text id="f1"
empty frontier ≠ global completion
```

---

# 12. True completion condition

Need BOTH:

```text id="f2"
queue empty
AND
no active expansions
```

That is what `join()` approximates.

---

# 13. Fixpoint visualization

```mermaid id="m4"
flowchart TD

    A[Current Frontier]
    --> B[Expand Nodes]

    B --> C{New Work Generated?}

    C -->|Yes| A

    C -->|No| D[Fixpoint Reached]
```

---

# 14. Mathematical interpretation

This is literally:

```text id="f3"
least fixpoint computation
```

The system repeatedly applies:

```text id="f4"
expand(frontier)
```

until:

```text id="f5"
expand(frontier) = frontier
```

This appears everywhere:

* graph traversal,
* Datalog,
* distributed systems,
* static analysis,
* recursive databases.

---

# 15. Sentinel Semantics

## Core idea

```text id="ss0"
Termination becomes a first-class message in the execution stream.
```

---

# 16. Why sentinel pattern is elegant

Instead of:

* killing threads,
* polling booleans,
* interrupts,

you reuse:

* existing synchronization channel.

Very compositional.

---

# 17. Sentinel visualization

```mermaid id="m5"
flowchart LR

    Q[Queue]
    -->|URL| W[Worker Processes]

    Q
    -->|None Sentinel| X[Worker Terminates]
```

---

# 18. Why one sentinel per worker?

Each sentinel only terminates:

```text id="ss1"
one consumer
```

So:

| Workers | Sentinels Needed |
| ------- | ---------------- |
| 1       | 1                |
| 4       | 4                |
| N       | N                |

---

# 19. Deep interpretation

Sentinel is effectively:

```text id="ss2"
a terminal graph edge
```

It propagates:

* lifecycle completion,
  through:
* the same causality channel as work.

---

# 20. Lifecycle Topology

This is the deepest abstraction of all.

## Core idea

```text id="lt0"
Correctness depends on how ownership and causality flow through system components.
```

---

# 21. Your crawler topology

You currently have:

```mermaid id="m6"
flowchart TD

    A[Seed URL]
    --> B[Frontier Queue]

    B --> C[Workers]

    C --> D[Expansion]

    D --> E[Filtering]

    E --> F[Dedup]

    F -->|new work| B

    B -->|sentinel| G[Termination]
```

---

# 22. Why topology matters

Concurrency correctness is not local.

It depends on:

* global structure of causality.

Examples:

| Bad topology             | Result                 |
| ------------------------ | ---------------------- |
| cyclic waits             | deadlock               |
| premature shutdown edges | lost work              |
| shared mutable ownership | races                  |
| blocking under lock      | serialization collapse |

---

# 23. Lifecycle topology questions

Every concurrent system implicitly answers:

| Question                        | Meaning         |
| ------------------------------- | --------------- |
| who owns work?                  | queue/frontier  |
| who owns completion?            | join/accounting |
| who owns shutdown?              | orchestrator    |
| who may generate work?          | workers         |
| when may termination propagate? | after fixpoint  |

---

# 24. Distributed systems interpretation

These same ideas scale directly to:

* Ray,
* Kafka,
* Airflow,
* distributed schedulers,
* actor systems.

Because they are fundamentally solving:

```text id="lt1"
global quiescence detection
under partial-order execution
```

---

# 25. Deep unification

These 5 concepts are really different views of one invariant:

| Concept            | Perspective                 |
| ------------------ | --------------------------- |
| queue accounting   | conservation of work        |
| shutdown ordering  | causality preservation      |
| fixpoint detection | completion proof            |
| sentinel semantics | lifecycle messaging         |
| lifecycle topology | global dependency structure |

---

# 26. Ultimate Compression

The entire worker-pool concurrency model can be compressed as:

```text id="final"
Maintain exact work conservation
while dynamically expanding a causal frontier
until a globally detectable fixpoint permits orderly termination propagation.
```

That sentence is surprisingly close to how real distributed execution engines are conceptualized internally.


Yes — you are now asking the *central invariant question*.

You correctly noticed:

```text id="q0"
queue.empty() is insufficient because workers may still generate descendants
```

That insight is exactly why `queue.join()` exists.

But importantly:

```text id="q1"
queue.join() does NOT mean "queue empty"
```

It means something much stronger.

---

# 1. What `queue.join()` ACTUALLY means

Python Queue tracks an internal counter:

```text id="q2"
unfinished_tasks
```

Mechanically:

| Event               | Counter Change |
| ------------------- | -------------- |
| `queue.put(x)`      | +1             |
| `queue.task_done()` | -1             |

Then:

```python id="q3"
queue.join()
```

blocks until:

```text id="q4"
unfinished_tasks == 0
```

---

# 2. Why this is stronger than queue empty

Suppose:

Queue empty,
BUT:

```text id="q5"
worker still processing item
```

Then:

* worker may still enqueue descendants.

So system NOT complete.

Queue emptiness alone misses:

* in-flight causality.

---

# 3. Example

Suppose:

```text id="q6"
unfinished_tasks = 1
```

because:

* one worker currently processing URL A.

Queue may already be:

```text id="q7"
[]
```

But worker might still discover:

* B,
* C,
* D.

So:

* queue empty,
* but frontier not exhausted.

---

# 4. What `join()` really proves

`join()` proves:

```text id="q8"
No queued work
AND
No active work
AND
No future descendants pending
```

because descendants must originate from:

* currently unfinished tasks.

This is the key invariant.

---

# 5. The fixpoint interpretation

This is literally:

```text id="q9"
global frontier exhaustion
```

or mathematically:

```text id="q10"
least fixpoint reached
```

---

# 6. Play out the fixpoint carefully

Suppose graph:

```text id="q11"
A → B,C
B → D
C → E
```

---

# 7. Initial state

Main thread:

```python id="q12"
queue.put(A)
```

Internal:

| State            | Value |
| ---------------- | ----- |
| queue            | [A]   |
| unfinished_tasks | 1     |

---

# 8. Worker processes A

Worker:

```python id="q13"
item = queue.get()
```

Queue now empty.

BUT:

```text id="q14"
unfinished_tasks still = 1
```

because:

* processing not completed.

---

# 9. Worker discovers B,C

Worker calls:

```python id="q15"
queue.put(B)
queue.put(C)
```

Now:

| State            | Value |
| ---------------- | ----- |
| queue            | [B,C] |
| unfinished_tasks | 3     |

Why 3?

| Task               | Count |
| ------------------ | ----- |
| A still processing | 1     |
| B enqueued         | 1     |
| C enqueued         | 1     |

---

# 10. Worker finishes A

Finally:

```python id="q16"
queue.task_done()
```

Now:

| unfinished_tasks | 2 |

Representing:

* B,
* C.

---

# 11. This continues recursively

Eventually:

| Queue            | empty |
| ---------------- | ----- |
| Workers          | idle  |
| unfinished_tasks | 0     |

ONLY THEN:

* `join()` returns.

---

# 12. The beautiful invariant

The key invariant is:

```text id="q17"
Any future work must descend causally
from an unfinished task.
```

Therefore:

```text id="q18"
unfinished_tasks == 0
⇒
no future work can ever appear
```

That is the deep proof.

---

# 13. Completion ownership

You asked:

> completion ownership is done within worker?

YES.

Very important insight.

---

# 14. Ownership model

| Responsibility        | Owner       |
| --------------------- | ----------- |
| create work           | producer    |
| process work          | worker      |
| declare work complete | SAME worker |

Critical invariant:

```text id="q19"
the consumer that got the task
owns completion accounting
```

Because only that worker knows:

```text id="q20"
when all descendants have been safely generated
```

---

# 15. Why task_done must happen AFTER expansion

This is extremely important.

Wrong:

```python id="q21"
task_done()
enqueue(children)
```

Because:

```text id="q22"
join() could falsely conclude global completion
before descendants inserted
```

That would violate:

* causality accounting.

---

# 16. Correct causality order

Correct:

```mermaid id="m1"
flowchart TD

    A[get task]
    --> B[expand descendants]

    B --> C[enqueue descendants]

    C --> D[task_done]
```

Meaning:

```text id="q23"
parent task remains unfinished
until descendants safely exist in accounting graph
```

Very deep invariant.

---

# 17. Why `for _ in range(max_workers)` exists

You asked:

> if allocation is not always max?

Good question.

Sentinels terminate:

* workers,
  not:
* tasks.

Each worker eventually blocks on:

```python id="q24"
queue.get()
```

To wake blocked workers:

* need one sentinel per blocked consumer.

---

# 18. Why usually one sentinel per worker

Suppose:

* 4 workers.

Eventually all become:

```text id="q25"
blocked consumers
```

Need:

* 4 wakeups.

Each sentinel only wakes:

* one blocked worker.

---

# 19. Why "actual active work count" doesn't matter

Because shutdown targets:

```text id="q26"
consumer loops
```

not:

* currently busy workers.

Even idle blocked workers need:

* a termination message.

---

# 20. Visualizing final stable fixpoint

```mermaid id="m2"
flowchart TD

    A[unfinished_tasks > 0]
    --> B[workers expanding graph]

    B --> C[enqueue descendants]

    C --> D[task_done parent]

    D --> E{unfinished_tasks == 0?}

    E -->|No| B

    E -->|Yes| F[Global Fixpoint]

    F --> G[Inject N Sentinels]

    G --> H[Workers Exit]
```

---

# 21. Deepest conceptual compression

The profound idea underneath `queue.join()` is:

```text id="final"
Completion is not absence of buffered work.
Completion is the impossibility of future work generation.
```

That is the true invariant being detected.


## LeetCode Exercise Critique

### 1. Complexity and Trade-offs of all solution attempts, with the main emphasis on the last attempt.

The notebook shows a clear progression from a broken queue-worker draft to a mostly correct concurrent crawler. The earliest attempts had correctness blockers: invalid `threading.lock` import, treating `htmlParser` like a dictionary instead of calling `getUrls`, missing `task_done()` symmetry, premature sentinel insertion, and returning/resetting state incorrectly. Those versions were not meaningfully analyzable beyond saying the intended shape was a shared frontier plus worker pool.

The final solution cell is much better. It now:
- filters by hostname via `urlparse(...).netloc`
- uses `htmlParser.getUrls(item)` correctly
- protects deduplication with a lock
- waits on `queue.join()` before injecting sentinels
- ensures `task_done()` runs through `finally`

For the last attempt, the intended time complexity is `O(V + E)` over same-host reachable pages and inspected outgoing links. Space is `O(V)` for `seen`, queued work, and worker coordination.

Main trade-offs in the last attempt:
- Good: the queue-centric worker model is the right concurrency topology for this problem.
- Good: moving `queue.put(url)` outside the lock avoids holding the critical section across a potentially blocking operation.
- Risk: debug `print(...)` inside `add_url_to_queue` materially distorts runtime under contention and would be harmful at scale.
- Risk: `max_queue_size` exists but is unused, so the implementation avoids bounded-queue liveness hazards only by silently becoming unbounded.
- Risk: the return type is a `set`; LeetCode accepts any order, but returning a list is a cleaner contract boundary.

Bottom line: the final attempt is close to interview-correct and passes the current harness, but it is still rough around interface cleanup, unnecessary state resets, and production-quality hygiene.

### 2. Critique of the problem-solving approach, including progression of thought and method.

Your progression was strong in one important sense: you kept converging on the correct systems model instead of thrashing between unrelated approaches. You consistently treated this as a frontier-expansion problem with shared deduplication, which is the right abstraction.

The evolution of thought looked like this:
- first: establish worker threads and a shared queue
- then: recognize that duplicate suppression must be synchronized
- then: fix parser interface usage and same-host filtering
- then: discover that queue accounting and shutdown ordering are separate invariants
- finally: separate lock-protected validation from queue insertion

What was good:
- You stayed with one architecture long enough to learn its invariants.
- You discovered the difference between queue emptiness and true global completion.
- You noticed that liveness bugs in concurrent code are often ordering bugs, not algorithmic-complexity bugs.

What still needs tightening:
- The code still carries exploratory scaffolding into the final version, especially debug prints and unused constructor parameters.
- The design is more stateful than necessary; `start_host`, `seen`, `queue`, and `lock` all live on `self` even though they are crawl-local.
- The final solution is interview-acceptable, but not yet a clean reusable component.

### 3. Improvements to Algorithm/ Optimal Example (include python solution code here in ``` ``` grouping braces)

The cleanest improvement is to keep crawl-specific state local to `crawl(...)`, minimize mutable object state, and preserve the exact two core invariants:
- dedupe before enqueue
- enqueue sentinels only after `queue.join()` proves all real work is done

```python
from concurrent.futures import ThreadPoolExecutor
from queue import Queue
from threading import Lock
from urllib.parse import urlparse


class Solution:
    def crawl(self, startUrl: str, htmlParser: 'HtmlParser'):
        host = urlparse(startUrl).netloc
        seen = {startUrl}
        queue = Queue()
        lock = Lock()
        workers = 8

        queue.put(startUrl)

        def try_schedule(url: str) -> None:
            if urlparse(url).netloc != host:
                return
            with lock:
                if url in seen:
                    return
                seen.add(url)
            queue.put(url)

        def worker() -> None:
            while True:
                url = queue.get()
                try:
                    if url is None:
                        return
                    for next_url in htmlParser.getUrls(url):
                        try_schedule(next_url)
                finally:
                    queue.task_done()

        with ThreadPoolExecutor(max_workers=workers) as executor:
            for _ in range(workers):
                executor.submit(worker)

            queue.join()

            for _ in range(workers):
                queue.put(None)

            queue.join()

        return list(seen)
```

Why this version is tighter:
- no stale shared executor on `self`
- no debug IO in the hot path
- explicit hostname filter boundary
- exact queue accounting preserved for both real work and sentinels

### 4. Applications in real-life situations, including AI-agent and engineering potential applications in 2026. Include examples from big tech and startups (frontier tech) for the exact problem and the generalized pattern. Be critical and outline tradeoffs, when to use this algorithm/design, and when not to use it.

Transferable systems pattern: concurrent frontier expansion over a graph with deduplicated work scheduling.

Literal usage versus analogy:
- Direct: internal web crawling, site map expansion, reachable-page discovery under one domain.
- Partial: service dependency exploration, data lineage traversal, document graph expansion.
- Conceptual: tool-graph exploration in agent systems where one action reveals more candidate actions.

What is its usefulness in designing large-scale data-driven applications?
- It gives a disciplined way to scale discovery workloads where new work appears during processing.
- It provides a natural place to enforce deduplication, backpressure, and completion detection.
- It is especially useful when correctness depends on not missing reachable descendants while still controlling concurrency.

Concrete examples:
- Big-tech-scale infrastructure example: a search/indexing platform crawls same-domain content trees, canonicalizes URLs, and parallelizes fetch/parse while deduplicating discovered links. The mapping is direct for the frontier-expansion part, but production systems add robots rules, retries, canonicalization, rate limits, and storage tiers.
- Startup/frontier-tech example: an enterprise knowledge startup expands document references, wiki links, and internal portals to construct retrieval corpora for downstream ranking. The mapping is partial: the queue/seen pattern transfers directly, while ranking, freshness, and access-control layers are additional systems.

Explicit 2026 AI-agent application mapping:
- Plausible use: an agent orchestration system explores reachable tools, docs, and subgoals from a seed task, deduplicating already-scheduled tool calls and using bounded worker concurrency to avoid overload. This is a conceptual but strong mapping.
- Do not use this approach when an agent workflow is small, sequential, and policy-constrained. In that case a deterministic planner with explicit step ordering is better than a concurrent frontier-expander.

Concise application case:
- Context and constraint: a platform must expand same-tenant documentation links quickly without crossing tenant boundaries.
- Algorithm/pattern choice: queue-centric concurrent frontier traversal with hostname or tenant-scope filtering plus exact deduplication.
- Decision and expected outcome: high IO overlap, no duplicate fetch scheduling, and predictable completion once the reachable frontier is exhausted.

```mermaid
flowchart LR
    Seed[Seed URL or seed task] --> Q[(Shared frontier queue)]
    Q --> W1[Worker 1]
    Q --> W2[Worker 2]
    W1 --> D[Discover descendants]
    W2 --> D
    D --> F{Within scope and unseen}
    F -- yes --> Q
    F -- no --> X[Discard]
```

When to use this design:
- dynamic discovery workloads
- IO-heavy tasks where parallel fetch latency matters
- problems requiring exact duplicate suppression

When not to use it:
- tiny traversals where a single-thread BFS is simpler
- workloads where all tasks are known upfront
- AI-agent pipelines where ordering, auditability, and policy gates matter more than throughput

### 5. Open Questions to Challenge My Understanding (non-spoiler). Ask 3-6 targeted questions tied to likely blind spots from my solution and reasoning.

1. In your final code, why is it safe to call `queue.put(url)` outside the lock after adding the URL to `seen`, and under what queue semantics would that become dangerous?
2. What exact property does `queue.join()` prove that `queue.empty()` does not?
3. Your final solution uses an unbounded `Queue`. If you reintroduced `max_queue_size`, what new liveness risk would appear and why?
4. Which state truly needs to live on `self`, and which state is logically scoped to a single `crawl(...)` invocation?
5. In your current implementation, what is the effect of leaving debug `print(...)` calls inside the synchronized hot path when the crawl fanout becomes large?

### 6. Next-Step Application Challenges (Similar but Variant) with Learning-Goal Intent. Provide 2-4 concise challenge prompts that are close to the current problem but differ in one key dimension (constraints, interface, mutability, streaming, memory, distributed setting, etc.). For each challenge include:

1. Variant: same crawler, but `getUrls(url)` may intermittently fail and should be retried up to two times.
   Learning goal intent: separate retry state from dedup state without corrupting completion detection.
   What changed from the original problem: processing one URL is no longer guaranteed to succeed on the first attempt.
   Why this change matters for design decisions: you must decide whether a failed URL remains scheduled, is re-enqueued, or moves to a retry buffer.

2. Variant: memory is capped, so `seen` cannot hold every visited URL exactly.
   Learning goal intent: reason about approximate deduplication and its correctness tradeoffs.
   What changed from the original problem: exact duplicate tracking is no longer free.
   Why this change matters for design decisions: false positives and external storage become architecture decisions rather than implementation details.

3. Variant: the crawl is distributed across multiple machines instead of one process.
   Learning goal intent: identify which invariants stop being local once queue and ownership become remote.
   What changed from the original problem: work scheduling and deduplication are now distributed systems problems.
   Why this change matters for design decisions: lock-based local reasoning no longer suffices; idempotency, leases, and ownership transfer become central.

4. Variant: instead of crawling URLs, each page yields more tool calls for a 2026 agent workflow, but execution order must remain auditable.
   Learning goal intent: compare throughput-oriented frontier expansion against policy-oriented orchestration.
   What changed from the original problem: traceability and ordered approval matter more than raw concurrency.
   Why this change matters for design decisions: a concurrent queue may no longer be the best primary abstraction; explicit plan graphs may be better.


## Further Improvements Toward a Production Search-Engine Crawler

Your current solution is a good interview-scale crawler core: shared frontier, deduplication, same-host filtering, worker concurrency, and correct queue lifecycle. A production search-engine crawler keeps that core idea, but expands it into a much larger distributed system.

### 1. Immediate improvements to this crawler

- remove debug `print(...)` from the hot path
- return `list(seen)` instead of a raw `set`
- keep crawl-local state inside `crawl(...)` instead of storing everything on `self`
- make worker count explicit and configurable
- add URL normalization before deduplication
- add tests for cycles, duplicate links, same-page self-links, and large fanout graphs

### 2. What production search crawlers add next

A real crawler for search engines usually adds these layers:

- URL canonicalization: normalize scheme, host casing, fragments, default ports, and duplicate path forms before deduping
- robots and policy enforcement: respect crawl permissions and per-site restrictions
- rate limiting and politeness: avoid overloading hosts; often one budget per domain or host group
- retry and error classification: distinguish transient network failures from permanent fetch failures
- prioritization: score URLs by freshness, importance, link signals, recrawl value, or sitemap hints
- persistence: store frontier, fetched metadata, and visited fingerprints outside process memory
- content hashing: avoid reprocessing identical content fetched from different URLs
- distributed ownership: shard hosts or URL hashes across machines so dedup and scheduling scale
- observability: metrics for queue depth, fetch latency, error rates, host saturation, and crawl coverage

### 3. Typical progression from interview design to production design

1. Single-process in-memory crawler
   This is your current stage.

2. Single-process crawler with normalized URLs and retry policy
   This is the first serious correctness upgrade.

3. Multi-process or multi-machine crawler with persistent frontier
   Now queue state and dedup state survive crashes.

4. Host-aware scheduler with politeness budgets
   This is where the crawler stops being just a graph traversal and becomes a network citizen.

5. Full search-engine ingestion pipeline
   Fetch, parse, canonicalize, dedup content, extract links, score for recrawl, and hand off documents to indexing.

### 4. Core architecture changes for a search engine crawler

The main shift is from:
- one in-memory shared queue

to:
- a persistent distributed frontier
- host-based scheduling
- durable deduplication state
- separate fetch, parse, and indexing stages

```mermaid
flowchart LR
    Seed[Seed URLs] --> Frontier[(Persistent frontier)]
    Frontier --> Scheduler[Host-aware scheduler]
    Scheduler --> Fetchers[Fetch workers]
    Fetchers --> Parser[Parser and link extractor]
    Parser --> Canonicalizer[URL canonicalization and dedup]
    Canonicalizer --> Frontier
    Parser --> DocStore[Content store]
    DocStore --> Indexer[Indexer and ranking pipeline]
```

### 5. The biggest conceptual upgrade

The interview problem is mostly about correctness of concurrency.

A production crawler is about balancing all of these at once:
- correctness
- politeness
- durability
- prioritization
- cost control
- recrawl strategy

So the next step is not just “make threads better.” It is to separate the crawler into explicit subsystems with durable ownership and scheduling policies.

### 6. Good next implementation steps if you want to keep leveling this notebook

- add URL normalization before `seen` checks
- add a retry counter per URL
- add host-level rate limiting
- replace in-memory `seen` with a durable key-value store abstraction
- replace one global queue with a scheduler that chooses the next eligible host
- add a second queue for parse/index work after fetch completes
